# 03 — Analyze recovery after major NBA injuries

This notebook reproduces the project's three decision metrics: same-body major reinjury, games played after confirmed return, and age-adjusted VORP change. The committed analysis extracts let this notebook run without repeating the network acquisition step.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 80)
pd.set_option('display.max_colwidth', 120)
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titleweight': 'bold',
    'axes.grid': True,
    'grid.alpha': 0.18,
    'font.size': 10,
})

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / 'data' / 'analysis'
DATA_DIR

In [ ]:
episodes = pd.read_csv(DATA_DIR / 'adjusted_multimetric_episode_data.csv', parse_dates=['Date'])
body = pd.read_csv(DATA_DIR / 'adjusted_outcomes_by_body_part.csv')
age = pd.read_csv(DATA_DIR / 'adjusted_outcomes_by_age.csv')
height = pd.read_csv(DATA_DIR / 'adjusted_outcomes_by_height.csv')
case_audit = pd.read_csv(DATA_DIR / 'adjusted_multimetric_30_case_audit.csv')
model_audit = pd.read_csv(DATA_DIR / 'adjusted_multimetric_model_audit.csv')

assert episodes['Episode ID'].is_unique
assert case_audit['Overall Automated Audit'].eq('PASS').all()

overview = pd.DataFrame({
    'Measure': [
        'Clean index surgery episodes',
        'Same-body major reinjuries',
        'Episodes with recovery metric',
        'Episodes with VORP metric',
        'Thirty-case audit passes',
    ],
    'Count': [
        len(episodes),
        int(episodes['Same-Body Major Reinjury'].sum()),
        int(episodes['Three-Year Games Played Pct'].notna().sum()),
        int(episodes['Age-Adjusted VORP Change'].notna().sum()),
        int(case_audit['Overall Automated Audit'].eq('PASS').sum()),
    ],
})
display(overview.style.hide(axis='index').format({'Count': '{:,.0f}'}))
display(model_audit)

## Modeling approach

The cohort contains distinct confirmed surgery episodes with known age and height and an observable three-year recurrence window. A major recurrence requires at least 30 missed games or defensible season-ending evidence.

Each adjusted estimate comes from a main-effects model containing body part, age bin, and height bin. Predictions for each displayed group are averaged over the observed distribution of the other two dimensions. Confidence intervals use player-cluster bootstrap resamples. These are descriptive standardized estimates, not causal effects.

VORP change compares actual three-season performance with the same player's age-expected path based on prior healthy production.

In [ ]:
COLORS = {'Reinjury': '#2F6BFF', 'Games': '#19A37B', 'VORP': '#E57A44'}

def add_bar_labels(ax, values, formatter):
    span = ax.get_xlim()[1] - ax.get_xlim()[0]
    for y, value in enumerate(values):
        ax.text(value + span * 0.015, y, formatter(value), va='center', fontsize=9)

def plot_body_part_bars(frame):
    labels = frame['Group'].tolist()
    y = np.arange(len(labels))
    fig, axes = plt.subplots(1, 3, figsize=(15, 6.2), constrained_layout=True)
    specs = [
        ('Adjusted Reinjury', 'Adjusted Reinjury 95% CI Low', 'Adjusted Reinjury 95% CI High', 'Same-body major reinjury', COLORS['Reinjury'], 100, lambda x: f'{x:.1f}%'),
        ('Adjusted Games', 'Adjusted Games 95% CI Low', 'Adjusted Games 95% CI High', 'Games played after return', COLORS['Games'], 100, lambda x: f'{x:.1f}%'),
        ('Adjusted VORP', 'Adjusted VORP 95% CI Low', 'Adjusted VORP 95% CI High', 'Age-adjusted VORP decline', COLORS['VORP'], -1, lambda x: f'{x:.2f}'),
    ]
    for ax, (value_col, low_col, high_col, title, color, multiplier, formatter) in zip(axes, specs):
        if value_col == 'Adjusted VORP':
            values = -frame[value_col].to_numpy()
            low = -frame[high_col].to_numpy()
            high = -frame[low_col].to_numpy()
            ax.axvline(0, color='0.35', linewidth=1)
        else:
            values = frame[value_col].to_numpy() * multiplier
            low = frame[low_col].to_numpy() * multiplier
            high = frame[high_col].to_numpy() * multiplier
        xerr = np.vstack([values - low, high - values])
        ax.barh(y, values, color=color, alpha=0.78, xerr=xerr, capsize=3)
        ax.set_yticks(y, labels)
        ax.invert_yaxis()
        ax.set_title(title)
        ax.grid(axis='y', visible=False)
        add_bar_labels(ax, values, formatter)
    fig.suptitle('Adjusted outcomes by body part', fontsize=15, fontweight='bold')
    return fig

def plot_three_outcome_lines(frame, title):
    x = np.arange(len(frame))
    labels = frame['Group'].tolist()
    fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True, constrained_layout=True)
    specs = [
        ('Adjusted Reinjury', 'Adjusted Reinjury 95% CI Low', 'Adjusted Reinjury 95% CI High', 'Same-body major reinjury (%)', COLORS['Reinjury'], 100, lambda v: f'{v:.1f}%'),
        ('Adjusted Games', 'Adjusted Games 95% CI Low', 'Adjusted Games 95% CI High', 'Games played after return (%)', COLORS['Games'], 100, lambda v: f'{v:.1f}%'),
        ('Adjusted VORP', 'Adjusted VORP 95% CI Low', 'Adjusted VORP 95% CI High', 'Age-adjusted VORP decline', COLORS['VORP'], -1, lambda v: f'{v:+.2f}'),
    ]
    for ax, (value_col, low_col, high_col, ylabel, color, multiplier, formatter) in zip(axes, specs):
        if value_col == 'Adjusted VORP':
            values = -frame[value_col].to_numpy()
            low = -frame[high_col].to_numpy()
            high = -frame[low_col].to_numpy()
            ax.axhline(0, color='0.35', linewidth=1)
        else:
            values = frame[value_col].to_numpy() * multiplier
            low = frame[low_col].to_numpy() * multiplier
            high = frame[high_col].to_numpy() * multiplier
        yerr = np.vstack([values - low, high - values])
        ax.errorbar(x, values, yerr=yerr, color=color, marker='o', linewidth=2, capsize=4)
        for px, py in zip(x, values):
            ax.annotate(formatter(py), (px, py), xytext=(0, 8), textcoords='offset points', ha='center', fontsize=9)
        ax.set_ylabel(ylabel)
        ax.grid(axis='x', visible=False)
    axes[-1].set_xticks(x, labels)
    axes[-1].set_xlabel('Group')
    fig.suptitle(title, fontsize=15, fontweight='bold')
    return fig

## Injury type

In [ ]:
plot_body_part_bars(body);

## Age

In [ ]:
plot_three_outcome_lines(age, 'Adjusted outcomes by age');

## Height

In [ ]:
plot_three_outcome_lines(height, 'Adjusted outcomes by height');

## Front-office interpretation

- **Injury type:** body part and procedure matter more than the generic major-injury label; ankle and foot injuries are the most concerning in this sample.
- **Height:** height is a recurrence-risk modifier, but not an automatic discount on availability or expected VORP.
- **Age:** age primarily changes the expected performance rebound. Younger players are more likely to regain their prior value after normal age-related availability differences are removed.

## Thirty-case audit

In [ ]:
audit_columns = [
    'Case Selection Reason', 'Player', 'Date', 'Body Group',
    'Same-Body Major Reinjury', 'Confirmed Return Date',
    'Three-Year Games Played Pct', 'Age-Adjusted VORP Change',
    'Overall Automated Audit',
]
audit_view = case_audit[audit_columns].copy()
audit_view['Three-Year Games Played Pct'] *= 100
display(audit_view.style.format({
    'Three-Year Games Played Pct': '{:.1f}%',
    'Age-Adjusted VORP Change': '{:+.2f}',
}).hide(axis='index'))

## Limitations

- The adjusted associations are descriptive, not causal.
- Recovery coverage is lower before 2010 because a confirmed appearance is required.
- Retirement and continued NBA employment affect both availability and VORP.
- Small groups can produce unstable estimates even after regularization.
- Same-body recurrence is broader than same-structure or same-side recurrence.